In [48]:
import nflreadpy as nfl
import polars as pl
import numpy as np
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from IPython.display import display, HTML

In [40]:
import polars as pl

model_data = pl.read_parquet(
    "../data/processed/model_data.parquet"
)

print(model_data.shape)

(2895, 110)


In [2]:
schedule_2026 = nfl.load_schedules(
    seasons=[2026]
)

schedule_2026 = schedule_2026.filter(
    pl.col("game_type") == "REG"
)

schedule_2026.select([
    "game_id",
    "week",
    "gameday",
    "gametime",
    "away_team",
    "home_team",
    "away_score",
    "home_score",
]).sort("week")

game_id,week,gameday,gametime,away_team,home_team,away_score,home_score
str,i32,str,str,str,str,i32,i32
"""2026_01_NE_SEA""",1,"""2026-09-09""","""20:20""","""NE""","""SEA""",10,13
"""2026_01_SF_LA""",1,"""2026-09-10""","""20:35""","""SF""","""LA""",27,7
"""2026_01_CHI_CAR""",1,"""2026-09-13""","""13:00""","""CHI""","""CAR""",59,37
"""2026_01_TB_CIN""",1,"""2026-09-13""","""13:00""","""TB""","""CIN""",27,33
"""2026_01_NO_DET""",1,"""2026-09-13""","""13:00""","""NO""","""DET""",30,31
…,…,…,…,…,…,…,…
"""2026_18_CHI_MIN""",18,"""2027-01-10""","""13:00""","""CHI""","""MIN""",null,null
"""2026_18_MIA_NE""",18,"""2027-01-10""","""13:00""","""MIA""","""NE""",null,null
"""2026_18_TB_NO""",18,"""2027-01-10""","""13:00""","""TB""","""NO""",null,null


In [3]:
completed_2026 = schedule_2026.filter(
    pl.col("home_score").is_not_null() &
    pl.col("away_score").is_not_null()
)

upcoming_2026 = schedule_2026.filter(
    pl.col("home_score").is_null() |
    pl.col("away_score").is_null()
)

In [4]:
print("Completed:", completed_2026.height)
print("Upcoming:", upcoming_2026.height)

Completed: 32
Upcoming: 240


In [5]:
upcoming_2026.select([
    "week",
    "gameday",
    "gametime",
    "away_team",
    "home_team",
]).sort([
    "week",
    "gameday"
])

week,gameday,gametime,away_team,home_team
i32,str,str,str,str
3,"""2026-09-24""","""20:15""","""ATL""","""GB"""
3,"""2026-09-27""","""13:00""","""LAC""","""BUF"""
3,"""2026-09-27""","""13:00""","""CAR""","""CLE"""
3,"""2026-09-27""","""13:00""","""NYJ""","""DET"""
3,"""2026-09-27""","""13:00""","""HOU""","""IND"""
…,…,…,…,…
18,"""2027-01-10""","""13:00""","""CHI""","""MIN"""
18,"""2027-01-10""","""13:00""","""MIA""","""NE"""
18,"""2027-01-10""","""13:00""","""TB""","""NO"""


In [6]:
pbp_2026 = nfl.load_pbp(
    seasons=[2026]
)

pbp_2026 = pbp_2026.filter(
    pl.col("season_type") == "REG"
)

print(pbp_2026.shape)

(5489, 372)


In [7]:
completed_game_ids = (
    completed_2026
    .select("game_id")
    .unique()
)

pbp_completed_2026 = (
    pbp_2026
    .join(
        completed_game_ids,
        on="game_id",
        how="inner"
    )
)

In [8]:
pbp_completed_2026.select([
    "game_id",
    "week",
    "home_team",
    "away_team"
]).unique().sort("week")

game_id,week,home_team,away_team
str,i32,str,str
"""2026_01_CLE_JAX""",1,"""JAX""","""CLE"""
"""2026_01_SF_LA""",1,"""LA""","""SF"""
"""2026_01_BAL_IND""",1,"""IND""","""BAL"""
"""2026_01_NE_SEA""",1,"""SEA""","""NE"""
"""2026_01_MIA_LV""",1,"""LV""","""MIA"""
…,…,…,…
"""2026_02_DET_BUF""",2,"""BUF""","""DET"""
"""2026_02_JAX_DEN""",2,"""DEN""","""JAX"""
"""2026_02_IND_KC""",2,"""KC""","""IND"""


In [9]:
home_games_2026 = completed_2026.select([
    "game_id",
    "week",
    "home_team",
    "away_team",
    pl.col("home_team").alias("team"),
    pl.col("home_score").alias("points_for"),
    pl.col("away_score").alias("points_against"),
    (pl.col("home_score") > pl.col("away_score"))
    .cast(pl.Int8)
    .alias("win"),
])

away_games_2026 = completed_2026.select([
    "game_id",
    "week",
    "home_team",
    "away_team",
    pl.col("away_team").alias("team"),
    pl.col("away_score").alias("points_for"),
    pl.col("home_score").alias("points_against"),
    (pl.col("away_score") > pl.col("home_score"))
    .cast(pl.Int8)
    .alias("win"),
])

team_games_2026 = (
    pl.concat([
        home_games_2026,
        away_games_2026
    ])
    .sort([
        "team",
        "week"
    ])
)

In [10]:
team_games_2026 = team_games_2026.with_columns([
    pl.col("win")
    .cum_sum()
    .over("team")
    .shift(1)
    .over("team")
    .alias("wins_before"),

    pl.col("points_for")
    .cum_sum()
    .over("team")
    .shift(1)
    .over("team")
    .alias("points_for_before"),

    pl.col("points_against")
    .cum_sum()
    .over("team")
    .shift(1)
    .over("team")
    .alias("points_against_before"),

    pl.col("win")
    .cum_count()
    .over("team")
    .shift(1)
    .over("team")
    .alias("games_before"),
])

In [11]:
team_games_2026 = team_games_2026.with_columns([
    (
        pl.col("wins_before") /
        pl.col("games_before")
    ).alias("win_pct_before"),

    (
        pl.col("points_for_before") /
        pl.col("games_before")
    ).alias("ppg_before"),

    (
        pl.col("points_against_before") /
        pl.col("games_before")
    ).alias("papg_before"),
])

In [12]:
team_games_2026 = team_games_2026.with_columns([
    pl.col("win")
    .shift(1)
    .rolling_mean(
        window_size=5,
        min_samples=1
    )
    .over("team")
    .alias("rolling_win_pct_5"),

    pl.col("points_for")
    .shift(1)
    .rolling_mean(
        window_size=5,
        min_samples=1
    )
    .over("team")
    .alias("rolling_ppg_5"),

    pl.col("points_against")
    .shift(1)
    .rolling_mean(
        window_size=5,
        min_samples=1
    )
    .over("team")
    .alias("rolling_papg_5"),

    (
        (pl.col("points_for") - pl.col("points_against"))
        .shift(1)
        .rolling_mean(
            window_size=5,
            min_samples=1
        )
        .over("team")
    ).alias("rolling_point_diff_5"),
])

In [13]:
team_games_2026.select([
    "team",
    "week",
    "wins_before",
    "games_before",
    "win_pct_before",
    "ppg_before",
    "papg_before",
    "rolling_win_pct_5",
    "rolling_ppg_5",
    "rolling_papg_5",
    "rolling_point_diff_5",
]).sort([
    "week",
    "team"
])

team,week,wins_before,games_before,win_pct_before,ppg_before,papg_before,rolling_win_pct_5,rolling_ppg_5,rolling_papg_5,rolling_point_diff_5
str,i32,i64,u32,f64,f64,f64,f64,f64,f64,f64
"""ARI""",1,null,null,null,null,null,null,null,null,null
"""ATL""",1,null,null,null,null,null,null,null,null,null
"""BAL""",1,null,null,null,null,null,null,null,null,null
"""BUF""",1,null,null,null,null,null,null,null,null,null
"""CAR""",1,null,null,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…
"""SEA""",2,1,1,1.0,13.0,10.0,1.0,13.0,10.0,3.0
"""SF""",2,1,1,1.0,27.0,7.0,1.0,27.0,7.0,20.0
"""TB""",2,0,1,0.0,27.0,33.0,0.0,27.0,33.0,-6.0


In [14]:
current_team_features_2026 = (
    team_games_2026
    .sort([
        "team",
        "week"
    ])
    .group_by("team")
    .last()
)

In [15]:
current_team_features_2026.select([
    "team",
    "week",
    "win_pct_before",
    "ppg_before",
    "papg_before",
    "rolling_win_pct_5",
    "rolling_ppg_5",
    "rolling_papg_5",
    "rolling_point_diff_5",
]).sort("team")

team,week,win_pct_before,ppg_before,papg_before,rolling_win_pct_5,rolling_ppg_5,rolling_papg_5,rolling_point_diff_5
str,i32,f64,f64,f64,f64,f64,f64,f64
"""ARI""",2,1.0,26.0,14.0,1.0,26.0,14.0,12.0
"""ATL""",2,0.0,13.0,20.0,0.0,13.0,20.0,-7.0
"""BAL""",2,1.0,41.0,23.0,1.0,41.0,23.0,18.0
"""BUF""",2,1.0,36.0,31.0,1.0,36.0,31.0,5.0
"""CAR""",2,0.0,37.0,59.0,0.0,37.0,59.0,-22.0
…,…,…,…,…,…,…,…,…
"""SEA""",2,1.0,13.0,10.0,1.0,13.0,10.0,3.0
"""SF""",2,1.0,27.0,7.0,1.0,27.0,7.0,20.0
"""TB""",2,0.0,27.0,33.0,0.0,27.0,33.0,-6.0


In [16]:
current_team_features_2026 = (
    team_games_2026
    .group_by("team")
    .agg([
        pl.col("win").sum().alias("wins"),
        pl.len().alias("games"),
        pl.col("points_for").sum().alias("points_for"),
        pl.col("points_against").sum().alias("points_against"),
    ])
    .with_columns([
        (
            pl.col("wins") /
            pl.col("games")
        ).alias("win_pct"),

        (
            pl.col("points_for") /
            pl.col("games")
        ).alias("ppg"),

        (
            pl.col("points_against") /
            pl.col("games")
        ).alias("papg"),
    ])
)

In [17]:
current_rolling_features_2026 = (
    team_games_2026
    .sort(["team", "week"])
    .group_by("team")
    .agg([
        pl.col("win")
        .tail(5)
        .mean()
        .alias("rolling_win_pct_5"),

        pl.col("points_for")
        .tail(5)
        .mean()
        .alias("rolling_ppg_5"),

        pl.col("points_against")
        .tail(5)
        .mean()
        .alias("rolling_papg_5"),

        (
            (pl.col("points_for") - pl.col("points_against"))
            .tail(5)
            .mean()
        ).alias("rolling_point_diff_5"),
    ])
)

In [18]:
current_team_features_2026 = (
    current_team_features_2026
    .join(
        current_rolling_features_2026,
        on="team",
        how="left"
    )
)

In [19]:
current_team_features_2026.sort("team")

team,wins,games,points_for,points_against,win_pct,ppg,papg,rolling_win_pct_5,rolling_ppg_5,rolling_papg_5,rolling_point_diff_5
str,i64,u32,i32,i32,f64,f64,f64,f64,f64,f64,f64
"""ARI""",1,2,33,45,0.5,16.5,22.5,0.5,16.5,22.5,-6.0
"""ATL""",0,2,16,54,0.0,8.0,27.0,0.0,8.0,27.0,-19.0
"""BAL""",1,2,58,47,0.5,29.0,23.5,0.5,29.0,23.5,5.5
"""BUF""",2,2,77,62,1.0,38.5,31.0,1.0,38.5,31.0,7.5
"""CAR""",1,2,71,62,0.5,35.5,31.0,0.5,35.5,31.0,4.5
…,…,…,…,…,…,…,…,…,…,…,…
"""SEA""",2,2,44,17,1.0,22.0,8.5,1.0,22.0,8.5,13.5
"""SF""",2,2,62,20,1.0,31.0,10.0,1.0,31.0,10.0,21.0
"""TB""",0,2,46,56,0.0,23.0,28.0,0.0,23.0,28.0,-5.0


In [20]:
week_3 = (
    upcoming_2026
    .filter(pl.col("week") == 3)
    .select([
        "game_id",
        "week",
        "gameday",
        "gametime",
        "away_team",
        "home_team",
    ])
)

In [21]:
home_current = current_team_features_2026.rename({
    "team": "home_team",
    "win_pct": "home_win_pct",
    "ppg": "home_ppg",
    "papg": "home_papg",
    "rolling_win_pct_5": "home_rolling_win_pct_5",
    "rolling_ppg_5": "home_rolling_ppg_5",
    "rolling_papg_5": "home_rolling_papg_5",
    "rolling_point_diff_5": "home_rolling_point_diff_5",
})

away_current = current_team_features_2026.rename({
    "team": "away_team",
    "win_pct": "away_win_pct",
    "ppg": "away_ppg",
    "papg": "away_papg",
    "rolling_win_pct_5": "away_rolling_win_pct_5",
    "rolling_ppg_5": "away_rolling_ppg_5",
    "rolling_papg_5": "away_rolling_papg_5",
    "rolling_point_diff_5": "away_rolling_point_diff_5",
})

In [22]:
week_3_features = (
    week_3
    .join(
        home_current,
        on="home_team",
        how="left"
    )
    .join(
        away_current,
        on="away_team",
        how="left"
    )
)

In [23]:
week_3_features = week_3_features.with_columns([
    (
        pl.col("home_win_pct") -
        pl.col("away_win_pct")
    ).alias("win_pct_diff"),

    (
        pl.col("home_ppg") -
        pl.col("away_ppg")
    ).alias("ppg_diff"),

    (
        pl.col("away_papg") -
        pl.col("home_papg")
    ).alias("defense_diff"),

    (
        pl.col("home_rolling_win_pct_5") -
        pl.col("away_rolling_win_pct_5")
    ).alias("rolling_win_pct_diff_5"),

    (
        pl.col("home_rolling_ppg_5") -
        pl.col("away_rolling_ppg_5")
    ).alias("rolling_ppg_diff_5"),

    (
        pl.col("away_rolling_papg_5") -
        pl.col("home_rolling_papg_5")
    ).alias("rolling_defense_diff_5"),

    (
        pl.col("home_rolling_point_diff_5") -
        pl.col("away_rolling_point_diff_5")
    ).alias("rolling_point_diff_diff_5"),
])

In [26]:
week_3_features.sort("gameday").select([
    "game_id",
    "gameday",
    "gametime",
    "away_team",
    "home_team",
    "home_win_pct",
    "away_win_pct",
    "win_pct_diff",
    "home_ppg",
    "away_ppg",
    "ppg_diff",
    "defense_diff",
    "rolling_win_pct_diff_5",
    "rolling_ppg_diff_5",
    "rolling_defense_diff_5",
    "rolling_point_diff_diff_5",
])

game_id,gameday,gametime,away_team,home_team,home_win_pct,away_win_pct,win_pct_diff,home_ppg,away_ppg,ppg_diff,defense_diff,rolling_win_pct_diff_5,rolling_ppg_diff_5,rolling_defense_diff_5,rolling_point_diff_diff_5
str,str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""2026_03_ATL_GB""","""2026-09-24""","""20:15""","""ATL""","""GB""",0.5,0.0,0.5,21.0,8.0,13.0,-1.0,0.5,13.0,-1.0,12.0
"""2026_03_LAC_BUF""","""2026-09-27""","""13:00""","""LAC""","""BUF""",1.0,0.0,1.0,38.5,14.0,24.5,-5.0,1.0,24.5,-5.0,19.5
"""2026_03_CAR_CLE""","""2026-09-27""","""13:00""","""CAR""","""CLE""",0.5,0.5,0.0,16.5,35.5,-19.0,4.5,0.0,-19.0,4.5,-14.5
"""2026_03_NYJ_DET""","""2026-09-27""","""13:00""","""NYJ""","""DET""",0.5,0.5,0.0,31.0,20.0,11.0,-20.5,0.0,11.0,-20.5,-9.5
"""2026_03_HOU_IND""","""2026-09-27""","""13:00""","""HOU""","""IND""",0.0,0.0,0.0,26.5,18.5,8.0,-9.0,0.0,8.0,-9.0,-1.0
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""2026_03_MIN_TB""","""2026-09-27""","""16:05""","""MIN""","""TB""",0.0,1.0,-1.0,23.0,24.0,-1.0,-15.5,-1.0,-1.0,-15.5,-16.5
"""2026_03_BAL_DAL""","""2026-09-27""","""16:25""","""BAL""","""DAL""",0.5,0.5,0.0,28.5,29.0,-0.5,-0.5,0.0,-0.5,-0.5,-1.0
"""2026_03_LV_NO""","""2026-09-27""","""16:25""","""LV""","""NO""",0.5,1.0,-0.5,27.0,26.5,0.5,-10.5,-0.5,0.5,-10.5,-10.0


In [27]:
offensive_plays_2026 = (
    pbp_completed_2026
    .filter(
        pl.col("posteam").is_not_null()
    )
)

In [28]:
offense_game_epa_2026 = (
    offensive_plays_2026
    .group_by([
        "game_id",
        "week",
        "posteam",
    ])
    .agg([
        pl.col("epa").sum().alias("off_epa"),
        pl.col("epa").mean().alias("off_epa_per_play"),
        pl.len().alias("off_plays"),

        pl.when(pl.col("pass_attempt") == 1)
        .then(pl.col("epa"))
        .otherwise(None)
        .mean()
        .alias("pass_epa_per_play"),

        pl.when(pl.col("rush_attempt") == 1)
        .then(pl.col("epa"))
        .otherwise(None)
        .mean()
        .alias("rush_epa_per_play"),
    ])
    .rename({
        "posteam": "team"
    })
)

In [29]:
defense_game_epa_2026 = (
    offensive_plays_2026
    .group_by([
        "game_id",
        "week",
        "defteam",
    ])
    .agg([
        pl.col("epa").sum().alias("def_epa_allowed"),
        pl.col("epa").mean().alias("def_epa_allowed_per_play"),
        pl.len().alias("def_plays"),
    ])
    .rename({
        "defteam": "team"
    })
)

In [30]:
team_game_epa_2026 = (
    offense_game_epa_2026
    .join(
        defense_game_epa_2026,
        on=["game_id", "week", "team"],
        how="inner"
    )
    .sort([
        "team",
        "week"
    ])
)

In [31]:
team_game_epa_2026.select([
    "game_id",
    "week",
    "team",
    "off_epa_per_play",
    "def_epa_allowed_per_play",
    "pass_epa_per_play",
    "rush_epa_per_play",
]).head(10)

game_id,week,team,off_epa_per_play,def_epa_allowed_per_play,pass_epa_per_play,rush_epa_per_play
str,i32,str,f64,f64,f64,f64
"""2026_01_ARI_LAC""",1,"""ARI""",0.026562,-0.227112,0.237181,-0.017525
"""2026_02_SEA_ARI""",2,"""ARI""",-0.259305,0.168459,-0.269476,-0.423636
"""2026_01_ATL_PIT""",1,"""ATL""",-0.305705,-0.211401,-0.606329,-0.081703
"""2026_02_CAR_ATL""",2,"""ATL""",-0.472802,-0.0108,-0.715734,-0.208286
"""2026_01_BAL_IND""",1,"""BAL""",0.156569,-0.11635,0.383099,0.179971
"""2026_02_NO_BAL""",2,"""BAL""",0.023849,0.13899,0.09757,0.136117
"""2026_01_BUF_HOU""",1,"""BUF""",0.259358,0.089056,0.454219,-0.124457
"""2026_02_DET_BUF""",2,"""BUF""",0.331139,0.226986,0.456024,0.272483
"""2026_01_CHI_CAR""",1,"""CAR""",0.077468,0.358259,0.251059,0.055131


In [32]:
current_epa_features_2026 = (
    team_game_epa_2026
    .group_by("team")
    .agg([
        pl.col("off_epa_per_play")
        .mean()
        .alias("off_epa_per_play"),

        pl.col("def_epa_allowed_per_play")
        .mean()
        .alias("def_epa_allowed_per_play"),

        pl.col("off_epa_per_play")
        .tail(5)
        .mean()
        .alias("rolling_off_epa_per_play_5"),

        pl.col("def_epa_allowed_per_play")
        .tail(5)
        .mean()
        .alias("rolling_def_epa_allowed_per_play_5"),

        pl.col("pass_epa_per_play")
        .tail(5)
        .mean()
        .alias("rolling_pass_epa_per_play_5"),

        pl.col("rush_epa_per_play")
        .tail(5)
        .mean()
        .alias("rolling_rush_epa_per_play_5"),
    ])
)

In [33]:
current_epa_features_2026.sort("team")

team,off_epa_per_play,def_epa_allowed_per_play,rolling_off_epa_per_play_5,rolling_def_epa_allowed_per_play_5,rolling_pass_epa_per_play_5,rolling_rush_epa_per_play_5
str,f64,f64,f64,f64,f64,f64
"""ARI""",-0.116372,-0.029327,-0.116372,-0.029327,-0.016148,-0.220581
"""ATL""",-0.389254,-0.1111,-0.389254,-0.1111,-0.661031,-0.144994
"""BAL""",0.090209,0.01132,0.090209,0.01132,0.240335,0.158044
"""BUF""",0.295249,0.158021,0.295249,0.158021,0.455122,0.074013
"""CAR""",0.033334,-0.057272,0.033334,-0.057272,0.271188,-0.087621
…,…,…,…,…,…,…
"""SEA""",0.087645,-0.145151,0.087645,-0.145151,0.381958,-0.118395
"""SF""",0.229134,-0.120099,0.229134,-0.120099,0.601202,0.078974
"""TB""",-0.038322,0.0421,-0.038322,0.0421,-0.30688,0.051775


In [34]:
home_epa_2026 = current_epa_features_2026.rename({
    "team": "home_team",
    "off_epa_per_play": "home_off_epa",
    "def_epa_allowed_per_play": "home_def_epa",
    "rolling_off_epa_per_play_5": "home_rolling_off_epa_5",
    "rolling_def_epa_allowed_per_play_5": "home_rolling_def_epa_5",
    "rolling_pass_epa_per_play_5": "home_rolling_pass_epa_5",
    "rolling_rush_epa_per_play_5": "home_rolling_rush_epa_5",
})

away_epa_2026 = current_epa_features_2026.rename({
    "team": "away_team",
    "off_epa_per_play": "away_off_epa",
    "def_epa_allowed_per_play": "away_def_epa",
    "rolling_off_epa_per_play_5": "away_rolling_off_epa_5",
    "rolling_def_epa_allowed_per_play_5": "away_rolling_def_epa_5",
    "rolling_pass_epa_per_play_5": "away_rolling_pass_epa_5",
    "rolling_rush_epa_per_play_5": "away_rolling_rush_epa_5",
})

In [35]:
week_3_features = (
    week_3_features
    .join(
        home_epa_2026,
        on="home_team",
        how="left"
    )
    .join(
        away_epa_2026,
        on="away_team",
        how="left"
    )
)

In [36]:
week_3_features = week_3_features.with_columns([
    (
        pl.col("home_off_epa") -
        pl.col("away_off_epa")
    ).alias("off_epa_diff"),

    (
        pl.col("away_def_epa") -
        pl.col("home_def_epa")
    ).alias("def_epa_diff"),

    (
        pl.col("home_rolling_off_epa_5") -
        pl.col("away_rolling_off_epa_5")
    ).alias("rolling_off_epa_diff_5"),

    (
        pl.col("away_rolling_def_epa_5") -
        pl.col("home_rolling_def_epa_5")
    ).alias("rolling_def_epa_diff_5"),

    (
        pl.col("home_rolling_pass_epa_5") -
        pl.col("away_rolling_pass_epa_5")
    ).alias("rolling_pass_epa_diff_5"),

    (
        pl.col("home_rolling_rush_epa_5") -
        pl.col("away_rolling_rush_epa_5")
    ).alias("rolling_rush_epa_diff_5"),
])

In [37]:
week_3_features.select([
    "game_id",
    "away_team",
    "home_team",
    "off_epa_diff",
    "def_epa_diff",
    "rolling_off_epa_diff_5",
    "rolling_def_epa_diff_5",
    "rolling_pass_epa_diff_5",
    "rolling_rush_epa_diff_5",
]).null_count()

game_id,away_team,home_team,off_epa_diff,def_epa_diff,rolling_off_epa_diff_5,rolling_def_epa_diff_5,rolling_pass_epa_diff_5,rolling_rush_epa_diff_5
u32,u32,u32,u32,u32,u32,u32,u32,u32
0,0,0,0,0,0,0,0,0


In [38]:
v3_features = [
    "win_pct_diff",
    "ppg_diff",
    "defense_diff",
    "rolling_win_pct_diff_5",
    "rolling_ppg_diff_5",
    "rolling_defense_diff_5",
    "rolling_point_diff_diff_5",
    "off_epa_diff",
    "def_epa_diff",
    "rolling_off_epa_diff_5",
    "rolling_def_epa_diff_5",
    "rolling_pass_epa_diff_5",
    "rolling_rush_epa_diff_5",
]

target = "home_win"

In [41]:
historical_v3 = (
    model_data
    .drop_nulls(subset=v3_features)
)

X_train_final = historical_v3.select(v3_features).to_numpy()
y_train_final = historical_v3.select(target).to_numpy().ravel()

In [42]:
print("Training games:", len(historical_v3))
print("Features:", len(v3_features))

Training games: 2602
Features: 13


In [45]:

final_rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    min_samples_leaf=10,
    random_state=42
)

final_rf.fit(
    X_train_final,
    y_train_final
)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",300
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",6
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",10
,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.Note: This parameter is tree-specific.",'gini'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=""sqrt""The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to `""sqrt""`.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",'sqrt'
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current node, ``N_t_L`` is the number of samples in theleft child, and ``N_t_R`` is the number of samples in the right child.``N``, ``N_t``, ``N_t_R`` and ``N_t_L`` all refer to the weighted sum,if ``sample_weight`` is passed... versionadded:: 0.19",0.0
,"bootstrap bootstrap

In [46]:
X_week3 = week_3_features.select(
    v3_features
).to_numpy()

week3_probabilities = final_rf.predict_proba(X_week3)[:, 1]

In [47]:
predictions = (
    week_3_features
    .select([
        "game_id",
        "gameday",
        "gametime",
        "away_team",
        "home_team",
    ])
    .with_columns([
        pl.Series(
            "home_win_probability",
            week3_probabilities
        )
    ])
    .with_columns([
        (1 - pl.col("home_win_probability"))
        .alias("away_win_probability"),

        pl.when(pl.col("home_win_probability") >= 0.5)
        .then(pl.col("home_team"))
        .otherwise(pl.col("away_team"))
        .alias("predicted_winner"),
    ])
    .sort("gameday")
)

predictions

game_id,gameday,gametime,away_team,home_team,home_win_probability,away_win_probability,predicted_winner
str,str,str,str,str,f64,f64,str
"""2026_03_ATL_GB""","""2026-09-24""","""20:15""","""ATL""","""GB""",0.798046,0.201954,"""GB"""
"""2026_03_LAC_BUF""","""2026-09-27""","""13:00""","""LAC""","""BUF""",0.748094,0.251906,"""BUF"""
"""2026_03_CAR_CLE""","""2026-09-27""","""13:00""","""CAR""","""CLE""",0.52803,0.47197,"""CLE"""
"""2026_03_NYJ_DET""","""2026-09-27""","""13:00""","""NYJ""","""DET""",0.571855,0.428145,"""DET"""
"""2026_03_HOU_IND""","""2026-09-27""","""13:00""","""HOU""","""IND""",0.634581,0.365419,"""IND"""
…,…,…,…,…,…,…,…
"""2026_03_MIN_TB""","""2026-09-27""","""16:05""","""MIN""","""TB""",0.389987,0.610013,"""MIN"""
"""2026_03_BAL_DAL""","""2026-09-27""","""16:25""","""BAL""","""DAL""",0.547068,0.452932,"""DAL"""
"""2026_03_LV_NO""","""2026-09-27""","""16:25""","""LV""","""NO""",0.441866,0.558134,"""LV"""


In [49]:
display_df = (
    predictions
    .with_columns([
        (
            pl.max_horizontal(
                "home_win_probability",
                "away_win_probability"
            )
            .alias("confidence")
        ),
        (
            pl.col("home_win_probability") * 100
        ).round(1).alias("home_win_pct"),
        (
            pl.col("away_win_probability") * 100
        ).round(1).alias("away_win_pct"),
    ])
    .select([
        "gameday",
        "gametime",
        "away_team",
        "home_team",
        "away_win_pct",
        "home_win_pct",
        "predicted_winner",
        "confidence",
    ])
)

In [50]:
display_df = display_df.with_columns(
    pl.when(pl.col("confidence") >= 0.75)
    .then(pl.lit("Very High"))
    .when(pl.col("confidence") >= 0.65)
    .then(pl.lit("High"))
    .when(pl.col("confidence") >= 0.55)
    .then(pl.lit("Moderate"))
    .otherwise(pl.lit("Close"))
    .alias("confidence_level")
)

In [51]:
display_df = display_df.with_columns([
    pl.col("away_win_pct")
    .cast(pl.String)
    .str.concat("%")
    .alias("away_win_pct"),

    pl.col("home_win_pct")
    .cast(pl.String)
    .str.concat("%")
    .alias("home_win_pct"),

    (
        pl.col("confidence") * 100
    )
    .round(1)
    .cast(pl.String)
    .str.concat("%")
    .alias("confidence"),
])

/var/folders/h2/qw5wsj9562xgw_1lbl6w8zgr0000gn/T/ipykernel_6121/2014119660.py:4: DeprecationWarning: `str.concat` is deprecated; use `str.join` instead. Note also that the default `delimiter` for `str.join` is an empty string, not a hyphen.
  .str.concat("%")
/var/folders/h2/qw5wsj9562xgw_1lbl6w8zgr0000gn/T/ipykernel_6121/2014119660.py:9: DeprecationWarning: `str.concat` is deprecated; use `str.join` instead. Note also that the default `delimiter` for `str.join` is an empty string, not a hyphen.
  .str.concat("%")
/var/folders/h2/qw5wsj9562xgw_1lbl6w8zgr0000gn/T/ipykernel_6121/2014119660.py:17: DeprecationWarning: `str.concat` is deprecated; use `str.join` instead. Note also that the default `delimiter` for `str.join` is an empty string, not a hyphen.
  .str.concat("%")


In [52]:
display_df

gameday,gametime,away_team,home_team,away_win_pct,home_win_pct,predicted_winner,confidence,confidence_level
str,str,str,str,str,str,str,str,str
"""2026-09-24""","""20:15""","""ATL""","""GB""","""20.2%25.2%47.2%42.8%36.5%38.0%…","""79.8%74.8%52.8%57.2%63.5%62.0%…","""GB""","""79.8%74.8%52.8%57.2%63.5%62.0%…","""Very High"""
"""2026-09-27""","""13:00""","""LAC""","""BUF""","""20.2%25.2%47.2%42.8%36.5%38.0%…","""79.8%74.8%52.8%57.2%63.5%62.0%…","""BUF""","""79.8%74.8%52.8%57.2%63.5%62.0%…","""High"""
"""2026-09-27""","""13:00""","""CAR""","""CLE""","""20.2%25.2%47.2%42.8%36.5%38.0%…","""79.8%74.8%52.8%57.2%63.5%62.0%…","""CLE""","""79.8%74.8%52.8%57.2%63.5%62.0%…","""Close"""
"""2026-09-27""","""13:00""","""NYJ""","""DET""","""20.2%25.2%47.2%42.8%36.5%38.0%…","""79.8%74.8%52.8%57.2%63.5%62.0%…","""DET""","""79.8%74.8%52.8%57.2%63.5%62.0%…","""Moderate"""
"""2026-09-27""","""13:00""","""HOU""","""IND""","""20.2%25.2%47.2%42.8%36.5%38.0%…","""79.8%74.8%52.8%57.2%63.5%62.0%…","""IND""","""79.8%74.8%52.8%57.2%63.5%62.0%…","""Moderate"""
…,…,…,…,…,…,…,…,…
"""2026-09-27""","""16:05""","""MIN""","""TB""","""20.2%25.2%47.2%42.8%36.5%38.0%…","""79.8%74.8%52.8%57.2%63.5%62.0%…","""MIN""","""79.8%74.8%52.8%57.2%63.5%62.0%…","""Moderate"""
"""2026-09-27""","""16:25""","""BAL""","""DAL""","""20.2%25.2%47.2%42.8%36.5%38.0%…","""79.8%74.8%52.8%57.2%63.5%62.0%…","""DAL""","""79.8%74.8%52.8%57.2%63.5%62.0%…","""Close"""
"""2026-09-27""","""16:25""","""LV""","""NO""","""20.2%25.2%47.2%42.8%36.5%38.0%…","""79.8%74.8%52.8%57.2%63.5%62.0%…","""LV""","""79.8%74.8%52.8%57.2%63.5%62.0%…","""Moderate"""
